In [5]:
import tkinter as tk
from tkinter import filedialog, messagebox
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np
import librosa
import joblib
import os
!pip install pydub
!pip install ffmpeg-python


   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   ---------------------------------------- 0/2 [future]
   -------------------- ------------------- 1/2 [ffmpeg-python]
   ---------------------------------------- 2/2 [ffmpeg-python]



In [6]:
model = joblib.load("voice_model.pkl")

scaler = joblib.load("voice_scaler.pkl")

encoder = joblib.load("voice_encoder.pkl")

print("Model,scaler and encoder loaded!")

Model,scaler and encoder loaded!


In [7]:
# RECORD AUDIO
def record_audio(filename="sound.wav", duration=3, fs=22050):
    label_status.config(text="Recording...")
    root.update()

    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='float32')
    sd.wait()

    write(filename, fs, audio)
    label_status.config(text=f"Recorded & Saved: {filename}")
    return filename

# MFCC EXTRACTION
def extract_mfcc(file_path, n_mfcc=20):
    audio, sample_rate = librosa.load(file_path)
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
    mfccs_mean = np.mean(mfccs.T, axis=0)
    return mfccs_mean.reshape(1, -1)

# TOP-3 ONLY
def predict_sound(path):
    mfcc = extract_mfcc(path)
    mfcc_scaled = scaler.transform(mfcc)

    probs = model.predict_proba(mfcc_scaled)[0]

    # Top 3
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3_conf = probs[top3_idx]
    top3_names = encoder.inverse_transform(top3_idx)

    return list(zip(top3_names, top3_conf))


# UPDATE BARS ONLY 
def update_bars(top3):
    names = [t[0] for t in top3]
    conf = [t[1] for t in top3]

    for canvas in bar_canvases:
        canvas.delete("all")

    for i in range(3):
        width = int(conf[i] * 250)
        color = ["blue", "green", "pink"][i]

        bar_canvases[i].create_rectangle(0, 0, width, 25, fill=color)
        bar_labels[i].config(text=f"{names[i]} — {conf[i]*100:.2f}%")


# RECORD & SHOW ONLY BARS
def record_and_predict():
    try:
        filepath = record_audio()
        top3 = predict_sound(filepath)
        update_bars(top3)

    except Exception as e:
        messagebox.showerror("Error", str(e))


# UPLOAD & SHOW ONLY BARS
def upload_and_predict():
    try:
        file_path = filedialog.askopenfilename(
            title="Select Audio File",
            filetypes=[("Audio Files", "*.wav *.mp3 *.M4A *.flac *.ogg")]
        )
        if not file_path:
            return

        label_status.config(text=f"Uploaded: {os.path.basename(file_path)}")
        root.update()

        top3 = predict_sound(file_path)
        update_bars(top3)

    except Exception as e:
        messagebox.showerror("Error", str(e))


# TKINTER UI
root = tk.Tk()
root.title("Voice Recognition App")
root.geometry("500x450")

tk.Label(root, text="✨ Namaste 🙏🌺", font=("poppous black", 18)).pack(pady=10)

btn_record = tk.Button(root, text="🎤 Record Voice (3 sec)", command=record_and_predict,
                       font=("Arial", 12), width=25)
btn_record.pack(pady=10)

btn_upload = tk.Button(root, text="📁 Upload Audio File", command=upload_and_predict,
                       font=("Arial", 12), width=25)
btn_upload.pack(pady=10)

label_status = tk.Label(root, text="Waiting for input...", font=("Arial", 12))
label_status.pack(pady=10)

# BARS
bar_labels = []
bar_canvases = []

for _ in range(3):
    lbl = tk.Label(root, text="---", font=("Arial", 12))
    lbl.pack()
    bar_labels.append(lbl)

    canvas = tk.Canvas(root, width=250, height=25, bg="white")
    canvas.pack(pady=4)
    bar_canvases.append(canvas)

root.mainloop()


C:\Users\HP\AppData\Local\Temp\ipykernel_10436\3889050512.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sample_rate = librosa.load(file_path)
C:\Users\HP\anaconda3\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
